# Anomaly detection and data quality for meter data

A retailer ingests meter data every day. Faults that get through — stuck meters, unit
changes, missing periods, impossible values — end up in forecasts, settlement and
customer bills. This notebook is the detector toolkit, applied to the half-hourly panel
in `../data/meter_halfhourly_2023.csv.gz`, which has several planted faults.

**What's in here**
- structural checks: duplicates, missing periods (DST-aware), completeness tables
- stuck values via run lengths
- unit changes and level shifts: ratio to trailing median, rolling-median change points, CUSUM
- spikes: robust z-scores (median / MAD) vs plain z-scores; profile-relative outliers
- physics checks: sign, capacity, night-time zeros
- distribution drift: KS test and a population-stability index
- multivariate: IsolationForest and LocalOutlierFactor on meter-day features
- treatment policy and the damage a fault does to a forecast
- a tidy `issues` table and a daily report function
- checklist

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)

raw = pd.read_csv("../data/meter_halfhourly_2023.csv.gz")
meters = pd.read_csv("../data/meters.csv").set_index("meter_id")

# settlement period -> UTC (see 02_pandas/09 for the reasoning): local midnight -> UTC, then + (period-1)*30 min
df = raw.drop_duplicates().copy()
df["utc"] = (pd.to_datetime(df["settlement_date"]).dt.tz_localize("Europe/London").dt.tz_convert("UTC")
             + pd.to_timedelta((df["settlement_period"] - 1) * 30, unit="min"))
df = df.sort_values(["meter_id", "utc"]).reset_index(drop=True)
df = df.join(meters[["customer_type", "has_solar"]], on="meter_id")
print(df.shape, "| duplicates removed:", len(raw) - len(df))
df.head(3)

(349399, 7) | duplicates removed: 40


,meter_id,settlement_date,settlement_period,kwh,utc,customer_type,has_solar
0,M100000,2023-01-01,1,0.734,2023-01-01 00:00:00+00:00,sme,False
1,M100000,2023-01-01,2,0.638,2023-01-01 00:30:00+00:00,sme,False
2,M100000,2023-01-01,3,0.394,2023-01-01 01:00:00+00:00,sme,False


## 1. Structural checks

Missing data is the first fault class and the easiest to miss, because nothing is
*wrong* in the rows you have. Compare against what *should* be there: 48 periods per
day, except 46 on 2023-03-26 and 50 on 2023-10-29.

In [2]:
expected = pd.Series(48, index=pd.date_range("2023-01-01", "2023-12-31", freq="D").strftime("%Y-%m-%d"))
expected["2023-03-26"], expected["2023-10-29"] = 46, 50

counts = df.groupby(["meter_id", "settlement_date"]).size().rename("n").reset_index()
grid = pd.MultiIndex.from_product([sorted(df["meter_id"].unique()), expected.index], names=["meter_id", "settlement_date"])
counts = counts.set_index(["meter_id", "settlement_date"]).reindex(grid, fill_value=0).reset_index()
counts["expected"] = counts["settlement_date"].map(expected)
counts["missing"] = counts["expected"] - counts["n"]

print("meter-days with missing periods:", (counts["missing"] > 0).sum(), "| fully missing days:", (counts["n"] == 0).sum())
counts[counts["n"] == 0].groupby("meter_id")["settlement_date"].agg(["min", "max", "count"])

meter-days with missing periods: 338 | fully missing days: 14


,min,max,count
meter_id,,,
M100011,2023-04-10,2023-04-23,14


In [3]:
# completeness by meter x month (share of expected periods present)
counts["month"] = counts["settlement_date"].str[:7]
completeness = counts.groupby(["meter_id", "month"]).apply(lambda g: g["n"].sum() / g["expected"].sum(), include_groups=False).unstack()
(completeness < 0.999).sum(axis=1).loc[lambda s: s > 0]

meter_id
M100000     4
M100001    10
M100002     6
M100003     7
M100004     3
M100005     3
M100006     5
M100007     3
M100008     5
M100009     4
M100010     2
M100011     4
M100012     7
M100013     1
M100014     6
M100015     8
M100016     4
M100017     9
M100018     2
M100019     4
dtype: int64

In [4]:
completeness.round(3).loc["M100011"]

month
2023-01    0.999
2023-02    1.000
2023-03    1.000
2023-04    0.533
2023-05    1.000
2023-06    0.999
2023-07    1.000
2023-08    0.999
2023-09    0.999
2023-10    0.999
2023-11    0.999
2023-12    1.000
Name: M100011, dtype: float64

**Interview check:** "M100011 is 97% complete for the year. Is that fine?" It depends
on *where* the 3% is: a fortnight of nothing (as here) invalidates April entirely, and
any rolling feature spanning the gap; 3% scattered across the year would be harmless.
Always look at the *structure* of missingness, not the share.

## 2. Stuck values

A meter that reports the same number for hours is a comms or firmware fault (the last
good value being re-sent). Detect with run lengths of identical consecutive readings
*within each meter*.

In [5]:
g = df.groupby("meter_id")["kwh"]
df["run_id"] = (df["kwh"] != g.shift()).cumsum()                     # new id whenever the value changes
runs = (df.groupby(["meter_id", "run_id"])
          .agg(value=("kwh", "first"), length=("kwh", "size"), start=("utc", "min"), end=("utc", "max"))
          .reset_index())
long_runs = runs[(runs["length"] >= 12) & (runs["value"] != 0)].sort_values("length", ascending=False)
print("runs of >= 12 identical non-zero readings:", len(long_runs))
long_runs.head()

runs of >= 12 identical non-zero readings: 1


,meter_id,run_id,value,length,start,end
59626,M100003,59627,0.187,336,2023-06-04 23:00:00+00:00,2023-06-11 22:30:00+00:00


**Pitfall:** `run_id` was built with a bare `shift()` inside `cumsum()`; the
`groupby(...).shift()` on the right-hand side is what prevents a run from bleeding across
the meter boundary. The `groupby(["meter_id", "run_id"])` then keeps the ids per meter.
Zero runs are excluded here because long zeros are legitimate for an empty property;
they get their own check below.

## 3. Unit changes and level shifts

A firmware update that starts sending Wh instead of kWh multiplies everything by 1000.
Compare each day's total with the meter's own trailing 28-day median.

In [6]:
# gross consumption (imports only): a solar meter's net daily total can be ~0 or negative in summer,
# which makes any ratio test meaningless -> test the import side, treat export separately
daily = (df.assign(kwh_gross=df["kwh"].clip(lower=0))
           .groupby(["meter_id", "settlement_date"])["kwh_gross"].sum().rename("kwh_day").reset_index())
daily["trailing_median"] = (daily.groupby("meter_id")["kwh_day"]
                                 .transform(lambda s: s.shift(1).rolling(28, min_periods=14).median()))
daily["ratio"] = daily["kwh_day"] / daily["trailing_median"]
level = daily[(daily["ratio"] > 10) | (daily["ratio"] < 0.1)]
print("days with ratio >10 or <0.1:", len(level))
level.groupby("meter_id").agg(first=("settlement_date", "min"), last=("settlement_date", "max"), n=("ratio", "size"),
                              min_ratio=("ratio", "min"), max_ratio=("ratio", "max")).round(2)

days with ratio >10 or <0.1: 29


,first,last,n,min_ratio,max_ratio
meter_id,,,,,
M100007,2023-09-01,2023-10-15,29,0.0,1116.26


The trailing median flags the *first* days of the fault (ratio ≈ 1000) and then, once
the window has filled with ×1000 values, the *first days after it ends* (ratio ≈ 0.001).
The middle of the fault looks normal to this test. Two change-point tools that see the
whole shift:

In [7]:
# (a) rolling-median change point: compare median of the next 7 days with the previous 7
def change_score(s, w=7):
    before = s.shift(1).rolling(w).median()
    after = s[::-1].rolling(w).median()[::-1]
    return np.log((after / before).clip(lower=1e-6))

daily["chg"] = daily.groupby("meter_id")["kwh_day"].transform(change_score)
cp = daily.loc[daily["chg"].abs() > np.log(5), ["meter_id", "settlement_date", "kwh_day", "chg"]]
cp.groupby("meter_id").agg(dates=("settlement_date", lambda s: f"{s.min()} .. {s.max()}"), n=("chg", "size"))

,dates,n
meter_id,,
M100007,2023-08-29 .. 2023-10-04,14


In [8]:
# (b) CUSUM on log daily energy: cumulative deviation from the meter's own mean drifts when the level shifts
def cusum(s):
    x = np.log(s.clip(lower=1e-3))
    return (x - x.mean()).cumsum()

daily["cusum"] = daily.groupby("meter_id")["kwh_day"].transform(cusum)
peak = daily.loc[daily.groupby("meter_id")["cusum"].transform(lambda c: c.abs() == c.abs().max())]
peak.sort_values("cusum", key=np.abs, ascending=False)[["meter_id", "settlement_date", "cusum"]].head(4)

,meter_id,settlement_date,cusum
2797,M100007,2023-08-31,-149.024450
4826,M100013,2023-04-06,34.730677
5208,M100014,2023-04-23,26.510409
6673,M100018,2023-04-28,26.442298


**Interview check:** "The ratio test flagged M100007 from 2023-09-01, but the CUSUM
extreme is on 2023-08-31. Are they consistent?" Yes. CUSUM accumulates deviations from
the *full-year* mean, which September inflates; every day before the fault is below that
mean, so the sum falls until the last clean day and climbs through September. The
extreme marks the change point; the sign says which way the level moved.

## 4. Spikes and point outliers

A plain z-score uses the mean and standard deviation, which the spike itself inflates,
so big spikes hide smaller ones. Use the rolling **median** and **MAD** (× 1.4826 to put
it on a standard-deviation scale).

In [9]:
def robust_z(s, w=48 * 7):
    med = s.rolling(w, min_periods=48, center=True).median()
    mad = (s - med).abs().rolling(w, min_periods=48, center=True).median() * 1.4826
    return (s - med) / mad.replace(0, np.nan)

df["z_plain"] = (df["kwh"] - g.transform("mean")) / g.transform("std")
df["z_robust"] = g.transform(robust_z)

for name in ["z_plain", "z_robust"]:
    flagged = df[df[name].abs() > 6]
    print(f"{name}: {len(flagged):5d} readings beyond |6|, in {flagged['meter_id'].nunique()} meters")

z_plain:   291 readings beyond |6|, in 18 meters
z_robust:  4154 readings beyond |6|, in 20 meters


In [10]:
# robust z flags far MORE readings: the MAD of a lognormal-ish meter is small, so its tails look extreme.
# A threshold is a calibration decision per meter, not a universal 6.
# The plain z-score for M100007 is dominated by September; robust z flags September AND is still usable elsewhere
m7 = df[df["meter_id"] == "M100007"]
print("M100007 std:", round(m7["kwh"].std(), 2), "| max z_plain outside Sept:", round(m7.loc[~m7["settlement_date"].str.startswith("2023-09"), "z_plain"].abs().max(), 3))
print("M100007 max z_robust outside Sept:", round(m7.loc[~m7["settlement_date"].str.startswith("2023-09"), "z_robust"].abs().max(), 1))

M100007 std: 59.63 | max z_plain outside Sept: 0.272
M100007 max z_robust outside Sept: 73.8


### Profile-relative outliers

A reading at 03:00 that would be normal at 18:00 is an anomaly. Compare each reading
with the meter's median for that **period of day and day type**, computed on the whole
year here (for a *monitoring* job that is fine; for a *forecasting feature* it would be
leakage).

In [11]:
df["daytype"] = np.where(df["utc"].dt.tz_convert("Europe/London").dt.dayofweek >= 5, "we", "wd")
key = ["meter_id", "settlement_period", "daytype"]
df["prof_med"] = df.groupby(key)["kwh"].transform("median")
df["prof_mad"] = df.groupby(key)["kwh"].transform(lambda s: (s - s.median()).abs().median() * 1.4826)
df["z_profile"] = (df["kwh"] - df["prof_med"]) / df["prof_mad"].replace(0, np.nan)

per_meter = df.groupby("meter_id")["z_profile"].apply(lambda z: (z.abs() > 8).mean() * 100).rename("% beyond |8|")
per_meter.round(2).sort_values(ascending=False).head(5)

meter_id
M100007    8.22
M100010    0.02
M100017    0.02
M100015    0.02
M100008    0.02
Name: % beyond |8|, dtype: float64

**Pitfall:** a global threshold on raw kWh ("flag anything above 5 kWh") flags every
SME all day and no residential meter ever. Thresholds must be per meter, or on a scaled
quantity like `z_profile`.

## 5. Physics checks

Things that cannot happen: negative consumption without solar, negative solar at night,
power above the supply capacity, an SME that is exactly zero at night for weeks (meter
off, not customer away).

In [12]:
local_hour = df["utc"].dt.tz_convert("Europe/London").dt.hour
daylight = local_hour.between(6, 20)
max_kwh_hh = 23 * 0.5            # 100 A single-phase supply ~= 23 kW -> 11.5 kWh per half hour (residential)

checks = pd.DataFrame({
    "negative_no_solar": (df["kwh"] < 0) & ~df["has_solar"],
    "negative_at_night": (df["kwh"] < 0) & ~daylight,
    "above_capacity_res": (df["kwh"] > max_kwh_hh) & (df["customer_type"] == "residential"),
    "sme_zero_at_night": (df["kwh"] == 0) & (df["customer_type"] == "sme") & ~daylight,
})
print(checks.sum())
df.loc[checks["above_capacity_res"], "meter_id"].value_counts().head(3)

negative_no_solar        0
negative_at_night        0
above_capacity_res    1439
sme_zero_at_night        0
dtype: int64


meter_id
M100007    1439
Name: count, dtype: int64

## 6. Distribution drift

Faults are not always spikes; sometimes the *distribution* moves (a new appliance, a
tenant change, a sensor drifting). Compare each meter-month with that meter's history.

In [13]:
df["month"] = df["settlement_date"].str[:7]

def psi(ref, cur, bins=10):
    edges = np.quantile(ref, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = -np.inf, np.inf
    p = np.histogram(ref, edges)[0] / len(ref) + 1e-6
    q = np.histogram(cur, edges)[0] / len(cur) + 1e-6
    return float(((q - p) * np.log(q / p)).sum())

rows = []
for mid, gm in df.groupby("meter_id"):
    for month, cur in gm.groupby("month"):
        hist = gm[gm["month"] < month]["kwh"]
        if len(hist) < 48 * 28:
            continue
        rows.append({"meter_id": mid, "month": month,
                     "ks_stat": stats.ks_2samp(hist, cur["kwh"]).statistic,
                     "psi": psi(hist.values, cur["kwh"].values)})
drift = pd.DataFrame(rows)
drift.sort_values("psi", ascending=False).head(6).round(3)

,meter_id,month,ks_stat,psi
84,M100007,2023-09,1.000,12.416
153,M100013,2023-12,0.434,3.199
152,M100013,2023-11,0.300,2.498
151,M100013,2023-10,0.166,1.293
5,M100000,2023-07,0.296,1.276
85,M100007,2023-10,0.116,1.273


Seasonality shows up as drift too (winter vs summer for every meter), so the useful
signal is a meter whose drift is far larger than its peers' in the same month:

In [14]:
drift["psi_rel"] = drift["psi"] / drift.groupby("month")["psi"].transform("median")
drift[drift["psi_rel"] > 5].round(2)

,meter_id,month,ks_stat,psi,psi_rel
84,M100007,2023-09,1.00,12.42,100.14
85,M100007,2023-10,0.12,1.27,8.37
143,M100013,2023-02,0.16,0.16,10.62
151,M100013,2023-10,0.17,1.29,8.50
152,M100013,2023-11,0.30,2.50,5.38


## 7. Multivariate anomalies on meter-day features

Turn each meter-day into a feature vector and let an unsupervised model rank them.
`IsolationForest` isolates points that are easy to separate; `LocalOutlierFactor`
compares local density. Neither knows what a fault *is*, so inspect the top hits.

In [15]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

night = ~daylight
feat = (df.assign(night_kwh=df["kwh"].where(night, 0.0), is_zero=(df["kwh"] == 0).astype(int))
          .groupby(["meter_id", "settlement_date"])
          .agg(total=("kwh", "sum"), peak=("kwh", "max"), night=("night_kwh", "sum"),
               n_zero=("is_zero", "sum"), n=("kwh", "size"), std=("kwh", "std")))
feat["night_share"] = feat["night"] / feat["total"].replace(0, np.nan)
feat["cv"] = feat["std"] / feat["total"].replace(0, np.nan) * feat["n"]
# max run length of identical values per meter-day
runlen = (df.groupby(["meter_id", "settlement_date", "run_id"]).size()
            .groupby(["meter_id", "settlement_date"]).max().rename("max_run"))
feat = feat.join(runlen).fillna(0)
# scale within meter so "big SME" is not an anomaly by itself
feat_z = feat.groupby("meter_id").transform(lambda c: (c - c.median()) / (c.std() + 1e-9))
cols = ["total", "peak", "night_share", "n_zero", "cv", "max_run", "n"]
X = StandardScaler().fit_transform(feat_z[cols].replace([np.inf, -np.inf], np.nan).fillna(0))

iso = IsolationForest(n_estimators=200, contamination=0.01, random_state=0).fit(X)
feat["iso_score"] = -iso.score_samples(X)                 # higher = more anomalous
feat["lof_score"] = -LocalOutlierFactor(n_neighbors=50).fit(X).negative_outlier_factor_
top = feat.sort_values("iso_score", ascending=False).head(12)
top[["total", "peak", "night_share", "n_zero", "max_run", "n", "iso_score", "lof_score"]].round(2)

total    peak  night_share  n_zero  max_run   n  iso_score  lof_score
meter_id settlement_date                                                                         
M100003  2023-06-06          8.98    0.19         0.38       0       48  48       0.71       3.28
         2023-06-07          8.98    0.19         0.38       0       48  48       0.71       3.28
         2023-06-11          8.98    0.19         0.38       0       48  48       0.71       3.28
         2023-06-10          8.98    0.19         0.38       0       48  48       0.71       3.28
         2023-06-09          8.98    0.19         0.38       0       48  48       0.71       3.28
         2023-06-08          8.98    0.19         0.38       0       48  48       0.71       3.28
         2023-06-05          8.98    0.19         0.38       0       48  48       0.71       3.28
M100014  2023-01-04         14.00    1.15         0.29       0        2  46       0.69       1.95
M100015  2023-11-01        125.64   14.68         0.12       0        2  48       0.69       2.03
M100007  2023-09-19       9236.00  472.00         0.34       0        2  47       0.68       2.17
         2023-09-02       8698.00  577.00         0.25       0        1  48       0.68       2.20
M100005  2023-07-03          5.47    0.61         0.30       0        2  46       0.67       2.06

In [16]:
def fault_label(idx):
    mid, d = idx
    if mid == "M100003" and "2023-06-05" <= d <= "2023-06-11": return "stuck"
    if mid == "M100007" and d.startswith("2023-09"): return "wh_units"
    if mid == "M100011" and "2023-04-10" <= d <= "2023-04-23": return "missing_fortnight"
    return ""
feat["planted"] = [fault_label(i) for i in feat.index]
n_top = 60
top60 = feat.sort_values("iso_score", ascending=False).head(n_top)
print(f"of the top {n_top} IsolationForest meter-days:")
print(top60["planted"].replace("", "none").value_counts())
print("planted fault meter-days in total:", (feat["planted"] != "").sum())

of the top 60 IsolationForest meter-days:
planted
none        42
wh_units    11
stuck        7
Name: count, dtype: int64
planted fault meter-days in total: 37


**Interview check:** "All seven stuck days rank at the very top, but only 11 of the 30
Wh-unit days make the top 60. The ×1000 fault is far bigger — why does it rank lower?"
Because the features were scaled *within meter*: a fault that lasts a month shifts
M100007's own median and standard deviation, so September looks less extreme relative
to itself than a single stuck week does relative to M100003. Robust scaling on a
reference period would fix it. Also inspect the "none" hits: many are days with 46 or
50 periods (`n` is a feature) or a genuinely high peak — plausible anomalies, not
faults. Detectors are complementary: rules for known signatures, unsupervised scores
for the unknown ones, and the `contamination` parameter is an assumption, not a
measurement.

## 8. Why it matters: the damage to a forecast

Fit a trivial per-meter forecast (tomorrow = mean of the same period over the last 7
days) with and without the faulty rows, and compare the error on clean days.

In [17]:
fault_mask = ((df["meter_id"] == "M100003") & df["settlement_date"].between("2023-06-05", "2023-06-11")) | \
             ((df["meter_id"] == "M100007") & df["settlement_date"].str.startswith("2023-09"))

def week_mean_forecast(frame):
    s = frame.groupby("meter_id")["kwh"]
    shifts = pd.concat([s.shift(48 * k) for k in range(1, 8)], axis=1)
    return shifts.mean(axis=1)                                     # mean of the same period on the previous 7 days, NaN-skipping

df["fc_dirty"] = week_mean_forecast(df)
df["fc_clean"] = week_mean_forecast(df.assign(kwh=df["kwh"].mask(fault_mask)))   # faulty rows masked before forecasting

after = (((df["meter_id"] == "M100003") & df["settlement_date"].between("2023-06-12", "2023-06-18")) |
         ((df["meter_id"] == "M100007") & df["settlement_date"].between("2023-10-01", "2023-10-07")))
df[after].groupby("meter_id")[["kwh", "fc_dirty", "fc_clean"]].apply(lambda t: pd.Series({
    "mae_dirty": (t["kwh"] - t["fc_dirty"]).abs().mean(),
    "mae_clean": (t["kwh"] - t["fc_clean"]).abs().mean()})).round(3)

,mae_dirty,mae_clean
meter_id,,
M100003,0.058,0.033
M100007,121.579,0.063


The table is the error in the **week after** each fault ends. For M100007 the dirty
forecast is off by hundreds of kWh per half hour for a week, and if the September rows
had been in a training set the model would have learned that September means ×1000. A
stuck week costs less but still biases the following week.

**Treatment policy** (what to do by fault class):

| Fault | Treatment | Never do |
|---|---|---|
| duplicate rows | drop exact duplicates; investigate conflicting ones | `mean()` them |
| missing periods | leave NaN; exclude from training; bill on estimate | `ffill` the target |
| stuck values | mask to NaN; flag meter for a comms check | treat as real zero-variance data |
| unit change | divide by 1000 **if confirmed**, else mask; escalate | "normalise" with a z-score |
| single spike | mask if above capacity; otherwise keep and use robust losses | clip silently |
| negative without solar | mask; check meter configuration | `abs()` |
| drift | flag; retrain per-meter model; investigate tenancy change | ignore because "the mean is fine" |

## 9. A tidy issues table and a daily report

Every detector writes into one long table with the same columns. That is what a
monitoring job produces and what a person triages.

In [18]:
issues = []
for _, r in counts[counts["missing"] > 0].iterrows():
    issues.append((r["meter_id"], r["settlement_date"], "missing_periods", "high" if r["n"] == 0 else "low", r["n"], r["expected"]))
for _, r in long_runs.iterrows():
    issues.append((r["meter_id"], str(r["start"].tz_convert("Europe/London").date()), "stuck_value", "high", r["length"], "< 12"))
for _, r in level.iterrows():
    issues.append((r["meter_id"], r["settlement_date"], "level_shift", "high", round(r["ratio"], 1), "~1"))
for _, r in df.loc[checks["above_capacity_res"]].iterrows():
    issues.append((r["meter_id"], r["settlement_date"], "above_capacity", "high", r["kwh"], f"<= {max_kwh_hh}"))
for _, r in df.loc[checks["negative_no_solar"]].iterrows():
    issues.append((r["meter_id"], r["settlement_date"], "negative_no_solar", "medium", r["kwh"], ">= 0"))
issues = pd.DataFrame(issues, columns=["meter_id", "date", "issue_type", "severity", "value", "expected"]).drop_duplicates(["meter_id", "date", "issue_type"])
print(issues.groupby(["issue_type", "severity"]).size())
issues.groupby("meter_id")["issue_type"].agg(lambda s: ", ".join(sorted(set(s)))).loc[lambda s: s.str.contains("stuck|level|missing_periods")].head(6)

issue_type       severity
above_capacity   high         30
level_shift      high         29
missing_periods  high         14
                 low         324
stuck_value      high          1
dtype: int64


meter_id
M100000                 missing_periods
M100001                 missing_periods
M100002                 missing_periods
M100003    missing_periods, stuck_value
M100004                 missing_periods
M100005                 missing_periods
Name: issue_type, dtype: object

In [19]:
def daily_report(issues, date):
    day = issues[issues["date"] == date]
    if day.empty:
        return f"{date}: no issues"
    lines = [f"{date}: {len(day)} issue(s) on {day['meter_id'].nunique()} meter(s)"]
    for (t, sev), grp in day.groupby(["issue_type", "severity"]):
        lines.append(f"  [{sev:6s}] {t:18s} {grp['meter_id'].nunique():3d} meters  e.g. {grp['meter_id'].iloc[0]} value={grp['value'].iloc[0]}")
    return "\n".join(lines)

print(daily_report(issues, "2023-09-01"))
print(daily_report(issues, "2023-06-06"))
print(daily_report(issues, "2023-04-12"))
print(daily_report(issues, "2023-02-14"))

2023-09-01: 5 issue(s) on 4 meter(s)
  [high  ] above_capacity       1 meters  e.g. M100007 value=107.0
  [high  ] level_shift          1 meters  e.g. M100007 value=1090.1
  [low   ] missing_periods      3 meters  e.g. M100003 value=47.0
2023-06-06: 2 issue(s) on 2 meter(s)
  [low   ] missing_periods      2 meters  e.g. M100006 value=47.0
2023-04-12: 2 issue(s) on 2 meter(s)
  [high  ] missing_periods      1 meters  e.g. M100011 value=0.0
  [low   ] missing_periods      1 meters  e.g. M100015 value=47.0
2023-02-14: no issues


## Checklist: data quality for energy time series

| Question | Detector |
|---|---|
| Are all expected rows there? | full grid reindex; DST-aware expected counts; completeness by meter × month |
| Any duplicates? Do they agree? | `duplicated()` on the key; compare values of conflicting duplicates |
| Is anything stuck? | run length of identical values per meter (`(x != x.groupby().shift()).cumsum()`) |
| Did the level jump? | daily total ÷ trailing median; rolling-median change score; CUSUM |
| Spikes? | robust z (median/MAD), per meter, profile-relative |
| Physically possible? | sign vs solar and daylight, capacity, night zeros |
| Distribution moved? | KS / PSI vs the meter's own history, relative to peers in the same month |
| Unknown unknowns? | IsolationForest / LOF on meter-day features, then *look* at the top hits |
| What do I do with it? | mask, never silently fix; exclude from training; escalate; log to an issues table |

**Pitfall:** every detector above was computed on the full year. For monitoring that is
right. For *features* in a forecasting model, per-meter medians and MADs must come from
the training period only.